# E2-04 · Pruebas χ²: independencia y bondad de ajuste

**Estadística II · Ingeniería Industrial · Universidad Industrial de Santander**  
*Versión docente (con soluciones)*

**Propósito de aprendizaje**

- Construir e interpretar tablas de contingencia con porcentajes por fila.
- Aplicar la prueba χ² de independencia, verificar sus condiciones y medir la fuerza de la asociación (V de Cramér).
- Identificar qué celdas explican la asociación con los residuos estandarizados.
- Usar la prueba χ² de bondad de ajuste para contrastar un modelo de Poisson con parámetro estimado.

**Duración sugerida:** 1 sesión de 2 horas + trabajo autónomo  
**Datos:** casos IC01 (despachos), IC09 (proyectos) e IC02 (interrupciones de planta)

## Estructura

| Parte | Caso | Modalidad | Pregunta |
|---|---|---|---|
| 1 | IC01 · Despachos | guiada | ¿El cumplimiento del despacho depende del turno? ¿Y de la zona del almacén? |
| 2 | IC09 · Proyectos | semi-guiada | ¿El cumplimiento del cronograma se asocia con la complejidad y con el esquema del equipo? |
| 3 | IC02 · Interrupciones | autónoma | ¿El número de interrupciones por turno sigue una distribución de Poisson? |

**Recordatorio.** En la prueba de independencia

$$\chi^2 = \sum_{i,j}\frac{(O_{ij}-E_{ij})^2}{E_{ij}}, \qquad E_{ij} = \frac{(\text{total fila } i)(\text{total columna } j)}{n}, \qquad gl = (f-1)(c-1).$$

La aproximación χ² es aceptable si ninguna frecuencia **esperada** es menor que 1 y no más del 20 % son menores que 5.

> **📌 Nota para el docente**
>
> **Resultados esperados con las bases del repositorio:**
> - **IC01 turno × cumplimiento:** χ² ≈ 40,9 (gl = 6), p < 0,001, V ≈ 0,19 (asociación débil a moderada). Una celda esperada < 5 (Madrugada–Retraso severo, E ≈ 3,8), es decir 1 de 12 celdas: se cumple la regla del 20 %. El turno de **noche** concentra los retrasos severos (≈ 21 % frente a ≈ 6 % en la mañana).
> - **IC01 zona × cumplimiento:** χ² ≈ 5,0 (gl = 8), p ≈ 0,76 → no hay evidencia de asociación. Sirve de contraste: no toda variable del proceso explica el cumplimiento.
> - **IC09:** complejidad × cronograma, χ² ≈ 189, V ≈ 0,43 (asociación fuerte); esquema × cronograma, χ² ≈ 25, V ≈ 0,16. Los proyectos tercerizados tienen más retrasos altos.
> - **IC02:** media ≈ 1,39 y varianza ≈ 2,11 interrupciones por turno. La sobredispersión (varianza > media) hace que el modelo de Poisson no ajuste bien: buen momento para hablar de la binomial negativa.

# 0. Preparación

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
sns.set_theme(style="whitegrid")
COLORES = ["#25A687", "#F2B134", "#FF6F49"]   # a tiempo, retraso moderado, retraso alto/severo
ALFA = 0.05

import os
from pathlib import Path

def cargar_csv(nombre, carpeta="casos_industriales/csv"):
    """Busca el archivo en la carpeta actual o en la carpeta datos/ del repositorio.
    Si no lo encuentra y se está en Google Colab, abre el diálogo para subirlo."""
    candidatos = [Path(nombre)] + [Path(p) / "datos" / carpeta / nombre for p in [".", "..", "../..", "../../.."]]
    for ruta in candidatos:
        if ruta.exists():
            print(f"Leyendo {ruta}")
            return pd.read_csv(ruta)
    try:
        from google.colab import files  # solo existe en Colab
        print(f"No encontré '{nombre}'. Súbalo desde su computador:")
        subido = files.upload()
        return pd.read_csv(next(iter(subido)))
    except ImportError:
        raise FileNotFoundError(f"No encontré '{nombre}'. Cópielo en la misma carpeta de este notebook.")


def chi2_independencia(tabla):
    """Prueba χ² de independencia con diagnóstico de condiciones y V de Cramér."""
    chi2, p, gl, esperadas = stats.chi2_contingency(tabla)
    n = tabla.to_numpy().sum()
    v = np.sqrt(chi2 / (n * (min(tabla.shape) - 1)))
    esperadas = pd.DataFrame(esperadas, index=tabla.index, columns=tabla.columns)
    print(f"χ² = {chi2:.3f}   gl = {gl}   p-valor = {p:.4g}   V de Cramér = {v:.3f}")
    print(f"Esperadas: mínima = {esperadas.min().min():.2f}; celdas con E < 5: {(esperadas < 5).sum().sum()} de {esperadas.size}")
    return chi2, p, gl, esperadas, v

---
# Parte 1 (guiada) · Cumplimiento de los despachos

El responsable de operaciones de un centro de distribución quiere saber si el cumplimiento de la hora de despacho está relacionado con el **turno** en que se preparó la orden y con la **zona** del almacén.

$$H_0: \text{el cumplimiento es independiente del turno} \qquad H_1: \text{existe asociación}$$

In [ ]:
despachos = cargar_csv("IC01_despachos_centro_distribucion.csv")
orden_cumpl = ["A tiempo", "Retraso moderado", "Retraso severo"]
orden_turno = ["Mañana", "Tarde", "Noche", "Madrugada"]
tabla_turno = pd.crosstab(despachos["turno_preparacion"], despachos["cumplimiento_despacho"]).loc[orden_turno, orden_cumpl]
print("Frecuencias observadas"); display(tabla_turno.assign(Total=tabla_turno.sum(axis=1)))
pct = 100 * tabla_turno.div(tabla_turno.sum(axis=1), axis=0)
print("Porcentaje por fila (cumplimiento dentro de cada turno)"); display(pct.round(1))
ax = pct.plot(kind="barh", stacked=True, color=COLORES, figsize=(9, 3.5))
ax.set_xlabel("% de las órdenes del turno"); ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left"); ax.invert_yaxis(); plt.show()

### 1.1 Prueba de independencia

Si H₀ fuera cierta, los porcentajes por fila serían parecidos en todos los turnos. La prueba compara lo observado con lo **esperado** bajo independencia.

In [ ]:
chi2, p, gl, esperadas, v = chi2_independencia(tabla_turno)
print("\nFrecuencias esperadas bajo independencia"); display(esperadas.round(1))

### 1.2 ¿Qué celdas explican la asociación?

El **residuo estandarizado ajustado** de cada celda,

$$r_{ij} = \frac{O_{ij}-E_{ij}}{\sqrt{E_{ij}\,(1-f_i/n)\,(1-c_j/n)}},$$

se comporta aproximadamente como una normal estándar bajo H₀. Valores con |r| > 2 señalan las celdas que se alejan de la independencia: positivos = más casos de los esperados.

In [ ]:
def residuos_ajustados(tabla, esperadas):
    n = tabla.to_numpy().sum()
    fi = tabla.sum(axis=1).to_numpy()[:, None] / n
    cj = tabla.sum(axis=0).to_numpy()[None, :] / n
    return (tabla - esperadas) / np.sqrt(esperadas * (1 - fi) * (1 - cj))

r = residuos_ajustados(tabla_turno, esperadas)
sns.heatmap(r, annot=True, fmt=".1f", cmap="RdBu_r", center=0, vmin=-5, vmax=5)
plt.title("Residuos estandarizados ajustados"); plt.show()

**Pregunta 1.1.** ¿Se cumplen las condiciones para usar la aproximación χ²? ¿Qué concluye sobre el turno y el cumplimiento? ¿Qué turno es el más problemático?

> **✅ Respuesta modelo (docente)**
>
> Sí: solo una de las 12 celdas tiene frecuencia esperada menor que 5 (Madrugada–Retraso severo, E ≈ 3,8), es decir, menos del 20 %, y ninguna es menor que 1. Con χ² ≈ 40,9 y 6 grados de libertad, p < 0,001: se rechaza la independencia, el cumplimiento **sí depende del turno**. La V de Cramér (≈ 0,19) indica una asociación débil a moderada. Los residuos muestran que el **turno de noche** tiene muchos más retrasos severos de los esperados (r ≈ +4,7) y muchos menos despachos a tiempo, mientras que la **mañana** tiene más despachos a tiempo de lo esperado. Las acciones de mejora deben empezar por el turno nocturno.

### 1.3 Contraste: zona del almacén

In [ ]:
tabla_zona = pd.crosstab(despachos["zona_almacen"], despachos["cumplimiento_despacho"])[orden_cumpl]
display((100 * tabla_zona.div(tabla_zona.sum(axis=1), axis=0)).round(1))
_ = chi2_independencia(tabla_zona)

**Pregunta 1.2.** ¿La zona del almacén se asocia con el cumplimiento? ¿Significa esto que la zona no importa para la operación?

> **✅ Respuesta modelo (docente)**
>
> No se rechaza H₀ (p ≈ 0,76): los porcentajes de cumplimiento son parecidos en las cinco zonas y no hay evidencia de asociación. Esto **no prueba** que la zona sea irrelevante (no rechazar no es lo mismo que demostrar independencia; con más datos podría detectarse un efecto pequeño), pero sí indica que, con la información disponible, la zona no explica los retrasos y que los esfuerzos deben dirigirse al turno.

---
# Parte 2 (semi-guiada) · Cumplimiento del cronograma en proyectos

La oficina de proyectos quiere saber qué características se asocian con el cumplimiento del cronograma: la **complejidad** del proyecto y el **esquema** con que se conformó el equipo.

**2.1** Construya la tabla de contingencia complejidad × cumplimiento (ordene las categorías), muestre los porcentajes por fila y aplique `chi2_independencia`.

In [ ]:
proyectos = cargar_csv("IC09_proyectos_industriales.csv")
orden_crono = ["A tiempo", "Retraso moderado", "Retraso alto"]
tabla_comp = pd.crosstab(proyectos["complejidad_proyecto"], proyectos["cumplimiento_cronograma"]).loc[["Baja", "Media", "Alta"], orden_crono]
display((100 * tabla_comp.div(tabla_comp.sum(axis=1), axis=0)).round(1))
chi2_c, p_c, gl_c, esp_c, v_c = chi2_independencia(tabla_comp)
sns.heatmap(residuos_ajustados(tabla_comp, esp_c), annot=True, fmt=".1f", cmap="RdBu_r", center=0); plt.title("Residuos ajustados"); plt.show()

**2.2** Repita el análisis para esquema del equipo × cumplimiento.

In [ ]:
tabla_esq = pd.crosstab(proyectos["esquema_equipo"], proyectos["cumplimiento_cronograma"]).loc[["Interno", "Mixto", "Tercerizado"], orden_crono]
display((100 * tabla_esq.div(tabla_esq.sum(axis=1), axis=0)).round(1))
chi2_e, p_e, gl_e, esp_e, v_e = chi2_independencia(tabla_esq)

**Pregunta 2.1.** Compare las dos asociaciones. ¿Cuál es más fuerte? ¿Qué recomendaría a la oficina de proyectos?

> **✅ Respuesta modelo (docente)**
>
> Las dos pruebas rechazan la independencia, pero la fuerza es muy distinta: complejidad × cronograma tiene V ≈ 0,43 (asociación fuerte) y esquema × cronograma V ≈ 0,16 (débil). Casi el 80 % de los proyectos de complejidad baja termina a tiempo, frente a menos del 10 % de los de complejidad alta, de los cuales cerca del 60 % tiene retraso alto. Recomendación: para proyectos de alta complejidad, planear holguras de cronograma más amplias, dividirlos en fases y reforzar el control de cambios de alcance. Los equipos tercerizados también muestran más retrasos altos, lo que sugiere revisar los mecanismos de supervisión de terceros. Como el estudio es observacional, estas asociaciones no prueban causalidad (por ejemplo, los proyectos complejos podrían tercerizarse con más frecuencia).

---
# Parte 3 (autónoma) · ¿Las interrupciones siguen un modelo de Poisson?

La planta registra el número de interrupciones no programadas por turno. El equipo de confiabilidad quiere usar un modelo de **Poisson** para planear la capacidad de respuesta. Contraste:

$$H_0: X \sim \text{Poisson}(\lambda) \qquad H_1: X \text{ no sigue una Poisson}$$

**Pasos sugeridos:**
1. Estime λ con la media muestral y compare la media con la varianza (en una Poisson son iguales).
2. Construya la tabla de frecuencias observadas y calcule las esperadas n·P(X = k). Agrupe la cola en una categoría "k o más" para que ninguna esperada sea menor que 5.
3. Aplique `stats.chisquare(obs, esp, ddof=1)`: se resta un grado de libertad adicional porque λ se estimó con los datos.
4. Grafique observadas vs. esperadas.

In [ ]:
planta = cargar_csv("IC02_consumos_planta.csv")
x = planta["numero_interrupciones"]
lam, n = x.mean(), len(x)
print(f"λ estimado = {lam:.3f}   varianza = {x.var():.3f}   índice de dispersión var/media = {x.var()/lam:.2f}")

corte = 4                                          # categorías 0, 1, 2, 3 y "4 o más"
k = np.arange(corte)
obs = np.append([(x == i).sum() for i in k], (x >= corte).sum())
esp = n * np.append(stats.poisson.pmf(k, lam), stats.poisson.sf(corte - 1, lam))
etiquetas = [str(i) for i in k] + [f"{corte} o más"]
display(pd.DataFrame({"observadas": obs, "esperadas": esp.round(2)}, index=etiquetas))
res = stats.chisquare(obs, esp, ddof=1)
print(f"χ² = {res.statistic:.3f}   gl = {len(obs) - 1 - 1}   p-valor = {res.pvalue:.4f}")

pos = np.arange(len(obs))
plt.bar(pos - .2, obs, .4, color="#00ADC6", label="observadas")
plt.bar(pos + .2, esp, .4, color="#FF6F49", label=f"esperadas Poisson({lam:.2f})")
plt.xticks(pos, etiquetas); plt.xlabel("interrupciones por turno"); plt.legend(); plt.show()

**Pregunta 3.1.** ¿El modelo de Poisson es adecuado? Relacione su conclusión con la comparación entre la media y la varianza. ¿Qué consecuencia práctica tendría usar Poisson si no ajusta?

> **✅ Respuesta modelo (docente)**
>
> La varianza (≈ 2,1) es claramente mayor que la media (≈ 1,4): hay **sobredispersión**. Hay más turnos sin interrupciones y más turnos con muchas interrupciones de lo que predice la Poisson, y menos turnos con valores intermedios. La prueba χ² (con 3 grados de libertad, después de agrupar la cola y restar el parámetro estimado) rechaza el ajuste. Si se usara Poisson, se **subestimaría** la probabilidad de turnos con muchas interrupciones y el equipo de respuesta quedaría corto justo en los peores turnos. Una alternativa es la binomial negativa, o analizar si las interrupciones dependen de alguna condición del turno (máquina, producto, operario) que genere la heterogeneidad.